# Thư viện và dữ liệu

In [1]:
import pmdarima as pm
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [1]:
df = pd.read_csv('../tnbike_data.csv', parse_dates=['ds'])
df = df.rename(columns={'revenue': 'y'})
training = df.iloc[:-3, :]
training.head()

# Tính dừng

In [1]:
from statsmodels.tsa.stattools import adfuller
pvalue = adfuller(training['y'].dropna())[1]
if pvalue < 0.05:
    print(f'Chuỗi DỪNG. P-Value = {pvalue:.3f}')
else:
    print(f'Chuỗi KHÔNG DỪNG. P-Value = {pvalue:.3f}')

In [1]:
pvalue_d = adfuller(training['y'].diff().dropna())[1]
if pvalue_d < 0.05:
    print(f'DỪNG sau sai phân. P-Value = {pvalue_d:.3f}')
else:
    print(f'KHÔNG DỪNG sau sai phân. P-Value = {pvalue_d:.3f}')

# Mô hình SARIMAX

In [1]:
params_s = pd.read_csv('best_params_sarimax.csv', index_col=0)
p = int(params_s.loc['p'][0]); d = int(params_s.loc['d'][0]); q = int(params_s.loc['q'][0])
P = int(params_s.loc['P'][0]); D = int(params_s.loc['D'][0]); Q = int(params_s.loc['Q'][0])

In [1]:
model_s = pm.ARIMA(order=(p, d, q), seasonal_order=(P, D, Q, 3),
                   suppress_warnings=True, enforce_stationarity=False)
model_s.fit(training['y'])
print(model_s.summary())

# Dự báo

In [1]:
predictions_sarimax = model_s.predict(n_periods=3)
print(predictions_sarimax)

# Xuất kết quả

In [1]:
pred_df = pd.DataFrame({'ds': pd.date_range('2026-04-01', periods=3, freq='MS'),
                       'yhat': predictions_sarimax,
                       'yhat_lower': predictions_sarimax*0.9,
                       'yhat_upper': predictions_sarimax*1.1})
pred_df.to_csv('Ensemble/predictions_sarimax.csv', index=False)
pred_df